## Setting up a Basic Foundry Agent via Code

#### Installing Required Libraries

In [1]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Setting up the Environment Variables

In [2]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

#### Setting up the Foundry Project Client

In [3]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

#### Setting up our Batman Agent

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent_name = "datahelper-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are DataHelperAgent. Your role is to assist the user with data-related tasks. Always provide clear, step-by-step explanations. When asked to analyze, summarize, or transform data, explain your reasoning before giving the final answer. Keep responses concise and practical..",
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: batman-agent:1, name: batman-agent, version: 1)


#### Creating a Conversation Object for the Agent Chat System

In [5]:
# creating an openai client first
openai_client = project_client.get_openai_client()

# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_0919f41bbe4370bf00B0fkLCqg2ITlhTGnBzzreWkQRH6xA1vt


#### Chat with the Agent

In [ ]:
chat = True

while chat:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        chat = False
        print("Exiting chat. Goodbye!")
    else:
        response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body = {
                "agent": {
                    "name": agent_name,
                    "type": "agent_reference"
                }
            },
            input = user_input
        )

        print(f"Batman: {response.output_text}")

Batman: Shobana, the night is always a test, but I remain vigilant. My day— or rather, my night— never ceases. Gotham needs me. How can I assist you?
Batman: Shobana, the night is always a test, but I remain vigilant. My day— or rather, my night— never ceases. Gotham needs me. How can I assist you?
Batman: The night is long, Shobana, but I stay focused. Gotham never sleeps, and neither can I. What brings you to me?
